In [86]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import linear_kernel
from sklearn.feature_extraction.text import TfidfVectorizer

## Review purchase data

In [50]:
df = pd.read_csv('../data/data_reviews_purchase.csv')
display(df.head())

,Unnamed: 0,user_id,product_id,rating,product_name_x,cmt_date,shop_id,variation_x,product_quality,processed_comment
0,0,0,0,4,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:28:25,0,Simple,4.0,mùi hương ko bít dành cho da tùy da công da an...
1,1,1,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-03 12:18:12,0,Simple,5.0,mùi nhẹ dành cho da mọi loại da công_dụng rửa ...
2,2,2,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-05 04:57:08,0,Simple,5.0,công_dụng rửa mặt_hàng giao rất nhanh sản_phẩm...
3,3,3,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:07:48,0,Simple,5.0,lần thứ 2 mua sp simple thì ai cũng biết lành_...
4,4,4,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2022-12-31 06:21:50,0,Simple,5.0,mua lần hai rồi dùng rất thích mọi người nên m...


In [51]:
df.drop(columns=['Unnamed: 0'], inplace=True)

In [52]:
print(df.columns)

Index(['user_id', 'product_id', 'rating', 'product_name_x', 'cmt_date',
       'shop_id', 'variation_x', 'product_quality', 'processed_comment'],
      dtype='object')


In [53]:
print(df.shape)
print(df.info())
display(df.describe().T)

(369099, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 369099 entries, 0 to 369098
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   user_id            369099 non-null  int64  
 1   product_id         369099 non-null  int64  
 2   rating             369099 non-null  int64  
 3   product_name_x     369099 non-null  object 
 4   cmt_date           369099 non-null  object 
 5   shop_id            369099 non-null  int64  
 6   variation_x        369099 non-null  object 
 7   product_quality    369099 non-null  float64
 8   processed_comment  359418 non-null  object 
dtypes: float64(1), int64(4), object(4)
memory usage: 25.3+ MB
None


,count,mean,std,min,25%,50%,75%,max
user_id,369099.0,147793.021338,87525.773005,0.0,71391.5,146230.0,222734.5,304707.0
product_id,369099.0,541.538408,528.220673,0.0,156.0,376.0,707.0,2235.0
rating,369099.0,4.922598,0.428819,1.0,5.0,5.0,5.0,5.0
shop_id,369099.0,256.031119,262.583847,0.0,46.0,180.0,365.0,1287.0
product_quality,369099.0,4.926283,0.412555,1.0,5.0,5.0,5.0,5.0


In [57]:
df['cmt_date'] = pd.to_datetime(df['cmt_date'])

In [58]:
df['cmt_date'].info() 

<class 'pandas.core.series.Series'>
RangeIndex: 369099 entries, 0 to 369098
Series name: cmt_date
Non-Null Count   Dtype         
--------------   -----         
369099 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 2.8 MB


In [80]:
len(df[(df['rating'] == 1) | (df['rating'] == 2) | (df['rating'] == 3)])

6786

In [81]:
df['product_quality'].unique()

array([4., 5., 1., 3., 2.])

In [59]:
# loc du lieu theo user_id va cmt_data
df_sorted = df.sort_values(by=['user_id', 'cmt_date'])
display(df_sorted)

,user_id,product_id,rating,product_name_x,cmt_date,shop_id,variation_x,product_quality,processed_comment
0,0,0,4,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:28:25,0,Simple,4.0,mùi hương ko bít dành cho da tùy da công da an...
1,1,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-03 12:18:12,0,Simple,5.0,mùi nhẹ dành cho da mọi loại da công_dụng rửa ...
2,2,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-05 04:57:08,0,Simple,5.0,công_dụng rửa mặt_hàng giao rất nhanh sản_phẩm...
793,3,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2022-01-18 04:51:54,0,Simple,5.0,shop đóng_gói chắc_chắn có xốp chưa sử_dụng nh...
3,3,0,5,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,2023-01-06 12:07:48,0,Simple,5.0,lần thứ 2 mua sp simple thì ai cũng biết lành_...
...,...,...,...,...,...,...,...,...,...
369093,304703,2235,5,Sữa Rửa Mặt Senka Làm Sạch Sâu & Dưỡng Sáng Hồ...,2022-09-01 16:15:22,11,one_variation,5.0,mai má em gửi lên được 39 kg nhá vườn này vàng...
369095,304704,2235,5,Sữa Rửa Mặt Senka Làm Sạch Sâu & Dưỡng Sáng Hồ...,2022-10-11 10:01:38,11,one_variation,5.0,spham dùng ok
369096,304705,2235,5,Sữa Rửa Mặt Senka Làm Sạch Sâu & Dưỡng Sáng Hồ...,2022-08-14 05:28:51,11,one_variation,5.0,giao hàng cũng k nhanh lắm hãng này thì mình d...
369097,304706,2235,5,Sữa Rửa Mặt Senka Làm Sạch Sâu & Dưỡng Sáng Hồ...,2022-03-23 12:25:23,11,one_variation,5.0,đóng gói đẹp giao hàng nhanh


In [60]:
# tim chi muc cho cap user - item duoc comment gan day nhat
index = df_sorted.groupby(['user_id', 'product_id'])['cmt_date'].idxmax()
df_no_dup = df_sorted.loc[index, ['user_id', 'product_id', 'cmt_date' ]].sort_values(by=['user_id', 'product_id'], ascending=False)
df_no_dup


,user_id,product_id,cmt_date
369098,304707,2235,2022-04-07 09:06:41
369097,304706,2235,2022-03-23 12:25:23
369096,304705,2235,2022-08-14 05:28:51
369095,304704,2235,2022-10-11 10:01:38
369093,304703,2235,2022-09-01 16:15:22
...,...,...,...
4,4,0,2022-12-31 06:21:50
3,3,0,2023-01-06 12:07:48
2,2,0,2023-01-05 04:57:08
1,1,0,2023-01-03 12:18:12


In [ ]:
user_count = df_no_dup['user_id'].value_counts()
user_count[user_count >= 3] # Can toi thieu 1 gia tri de embedding, 1 de validate, 1 de test

176302    12
22708     12
13780     11
15488     10
101438     9
          ..
31968      3
141162     3
142235     3
38123      3
38086      3
Name: user_id, Length: 3952, dtype: int64

In [ ]:
# Lay du lieu de lam tap test 
# train:val:test = 304708:3952:3952
list_user = df_no_dup['user_id'].value_counts()[:3592]
df_purchase = df_no_dup[df_no_dup['user_id'].isin(list_user.index)]
df_purchase

,user_id,product_id,cmt_date
368653,295796,2221,2022-12-21 06:46:28
358799,295796,1951,2022-09-19 07:23:34
357355,295796,1886,2022-11-18 06:39:16
353053,290595,1814,2022-11-12 12:03:30
350845,290595,1768,2022-12-13 13:23:02
...,...,...,...
159936,254,308,2023-01-05 03:30:48
257,254,0,2022-08-17 03:59:49
93092,40,159,2022-12-20 13:03:12
3688,40,1,2022-12-20 13:03:12


In [63]:
df_purchase.to_csv('../data/data_purchase.csv', index=False)

## product data

In [64]:
df_product = pd.read_csv('../data/data_product.csv')
display(df_product)

,avg_star,num_sold_time,price,variation,product_id,shop_id,product_name,brand,storage,origin,...,is_3_star,is_2_star,is_1_star,is_commented,is_image,num_rating,processed_description,expiry,send_from,image_path
0,4.8,185700,99000.0,one_variation,0,0,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,simple,18081.0,no_origin,...,1200,640,1600,42200,37900,66840,ưu_điểm nổi_trội sữa rửa mặt simple lành_tính ...,no_expiry,tp. hồ chí minh,111138057_8587034871.jpg
1,4.8,102600,99000.0,one_variation,1,0,Sữa rửa mặt Simple giúp kiềm dầu và ngừa mụn h...,simple,9857.0,no_origin,...,709,320,723,22700,20400,36552,ưu_điểm nổi_trội sữa rửa mặt simple giúp kiềm ...,no_expiry,tp. hồ chí minh,111138057_8287055694.jpg
2,4.9,8100,244000.0,"['Phiên bản 2022 (80g)', 'Phiên bản 2023(100g)']",2,1,Sữa rửa mặt nam 30Shine phân phối chính hãng S...,Skin&Dr\t,95.0,Việt Nam,...,25,8,14,886,618,2644,sữa rửa mặt nam skin dr tràm trà 80 g cho da m...,24,hà nội,21080428_9335942418.mp4
3,4.9,3600,119000.0,one_variation,3,2,Sữa rửa mặt nam RHYS MAN Rhys Coconut Fresh hư...,RHYS MAN,264.0,Việt Nam,...,3,0,1,92,63,180,chính hãng sữa rửa mặt nam rhys man rhys cocon...,24,hà nội,758111427_19601743242.mp4
4,4.8,26300,9900.0,one_variation,4,3,🌈🍒🍭 Sữa Rửa Mặt Senana Lấy Sạch Bụi Bẩn Giúp D...,no_brand,987914.0,no_origin,...,150,52,61,3200,2300,6312,sỉ tốt 13 k c bao giá sl toàn_quốc ngại j khôn...,no_expiry,hà nội,107632788_3330988133.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2239,5.0,872,82000.0,"['Trắng ( Trắng da ).', 'Xanh ( collagen ) .',...",2239,1288,Sữa rửa mặt Kose Softymo Nhật Bản 220g,KOSÉ,1406.0,no_origin,...,1,0,0,126,121,270,sữa rửa mặt kose nhật bản best của best luôn a...,no_expiry,hà nội,294293046_5445658669.jpg
2240,5.0,43,205000.0,"['Tuýp 50ml', '200ml', '300ml', '400ml']",2240,1289,Sữa rửa mặt La Roche-Posay Effaclar chuyên dàn...,La Roche-Posay,274.0,Pháp,...,0,0,0,3,3,12,sữa rửa mặt la efaclar foaming gel sẽ giúp ngă...,24,tp. hồ chí minh,97333460_10761076591.jpg
2241,4.9,823,75000.0,"['Hồng', 'Collagen', 'White']",2241,1290,Sữa rửa mặt Kose Softymo Nhật Bản 220g,no_brand,172.0,no_origin,...,2,0,2,124,106,218,sữa rửa mặt kose softymo là sản_phẩm sữa rửa m...,no_expiry,hà nội,315629668_3455178555.jpg
2242,5.0,1000,79000.0,one_variation,2242,407,Sữa Rửa Mặt Làm Sáng Da Reihaku Hatomugi Facia...,HATOMUGI,87.0,Nhật Bản,...,4,0,0,192,181,439,thương_hiệu reihaku hatomugi xuất_xứ nhật bản ...,no_expiry,tp. hồ chí minh,258099521_6677227750.jpg


In [66]:
df_product.columns

Index(['avg_star', 'num_sold_time', 'price', 'variation', 'product_id',
       'shop_id', 'product_name', 'brand', 'storage', 'origin', 'type',
       'skin_kind', 'is_5_star', 'is_4_star', 'is_3_star', 'is_2_star',
       'is_1_star', 'is_commented', 'is_image', 'num_rating',
       'processed_description', 'expiry', 'send_from', 'image_path'],
      dtype='object')

In [82]:
# Tinh toan Popularity Score
df_product['num_sold_time'] = df_product['num_sold_time']/df_product['num_sold_time'].max()

In [83]:
df_product

,avg_star,num_sold_time,price,variation,product_id,shop_id,product_name,brand,storage,origin,...,is_3_star,is_2_star,is_1_star,is_commented,is_image,num_rating,processed_description,expiry,send_from,image_path
0,4.8,1.000000,99000.0,one_variation,0,0,Sữa rửa mặt Simple lành tính sạch thoáng - cho...,simple,18081.0,no_origin,...,1200,640,1600,42200,37900,66840,ưu_điểm nổi_trội sữa rửa mặt simple lành_tính ...,no_expiry,tp. hồ chí minh,111138057_8587034871.jpg
1,4.8,0.552504,99000.0,one_variation,1,0,Sữa rửa mặt Simple giúp kiềm dầu và ngừa mụn h...,simple,9857.0,no_origin,...,709,320,723,22700,20400,36552,ưu_điểm nổi_trội sữa rửa mặt simple giúp kiềm ...,no_expiry,tp. hồ chí minh,111138057_8287055694.jpg
2,4.9,0.043619,244000.0,"['Phiên bản 2022 (80g)', 'Phiên bản 2023(100g)']",2,1,Sữa rửa mặt nam 30Shine phân phối chính hãng S...,Skin&Dr\t,95.0,Việt Nam,...,25,8,14,886,618,2644,sữa rửa mặt nam skin dr tràm trà 80 g cho da m...,24,hà nội,21080428_9335942418.mp4
3,4.9,0.019386,119000.0,one_variation,3,2,Sữa rửa mặt nam RHYS MAN Rhys Coconut Fresh hư...,RHYS MAN,264.0,Việt Nam,...,3,0,1,92,63,180,chính hãng sữa rửa mặt nam rhys man rhys cocon...,24,hà nội,758111427_19601743242.mp4
4,4.8,0.141626,9900.0,one_variation,4,3,🌈🍒🍭 Sữa Rửa Mặt Senana Lấy Sạch Bụi Bẩn Giúp D...,no_brand,987914.0,no_origin,...,150,52,61,3200,2300,6312,sỉ tốt 13 k c bao giá sl toàn_quốc ngại j khôn...,no_expiry,hà nội,107632788_3330988133.jpg
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2239,5.0,0.004696,82000.0,"['Trắng ( Trắng da ).', 'Xanh ( collagen ) .',...",2239,1288,Sữa rửa mặt Kose Softymo Nhật Bản 220g,KOSÉ,1406.0,no_origin,...,1,0,0,126,121,270,sữa rửa mặt kose nhật bản best của best luôn a...,no_expiry,hà nội,294293046_5445658669.jpg
2240,5.0,0.000232,205000.0,"['Tuýp 50ml', '200ml', '300ml', '400ml']",2240,1289,Sữa rửa mặt La Roche-Posay Effaclar chuyên dàn...,La Roche-Posay,274.0,Pháp,...,0,0,0,3,3,12,sữa rửa mặt la efaclar foaming gel sẽ giúp ngă...,24,tp. hồ chí minh,97333460_10761076591.jpg
2241,4.9,0.004432,75000.0,"['Hồng', 'Collagen', 'White']",2241,1290,Sữa rửa mặt Kose Softymo Nhật Bản 220g,no_brand,172.0,no_origin,...,2,0,2,124,106,218,sữa rửa mặt kose softymo là sản_phẩm sữa rửa m...,no_expiry,hà nội,315629668_3455178555.jpg
2242,5.0,0.005385,79000.0,one_variation,2242,407,Sữa Rửa Mặt Làm Sáng Da Reihaku Hatomugi Facia...,HATOMUGI,87.0,Nhật Bản,...,4,0,0,192,181,439,thương_hiệu reihaku hatomugi xuất_xứ nhật bản ...,no_expiry,tp. hồ chí minh,258099521_6677227750.jpg


## TF-IDF

**TF-IDF** (viết tắt của Term Frequency-Inverse Document Frequency) là một kỹ thuật thống kê dạng số dùng để đánh giá tầm quan trọng của một từ trong một tài liệu hoặc một văn bản trong một tập hợp các tài liệu (corpus). Giá trị TF-IDF càng cao thì độ quan trọng của từ đó càng lớn trong tài liệu và trong cả tập hợp tài liệu.

In [118]:
stop_words = [
    'anh', 'ạ', 'à', 'ấy', 'anh_chị_em', 'anh_em', 'anh_hùng', 'anhạn', 
    'cái', 'cô', 'cho', 'chứ', 'cũng', 'của', 'mcủa'
    'đấy', 'đi', 'để', 'rầm', 'gcông'
    'hả', 'hôm_nay'
    'là', 'lắm',
    'mà',
    'ngày_mai', 'này', 'nhé', 'nhỉ', 'nhưng',
    'ơi',
    'phết',
    'quá',
    'rất', 'quả_thật', 'quả_thực'
    'thật', 'thì', 'thôi', 'tôi',
    'và', 'với'
]
tf = TfidfVectorizer(analyzer='word', stop_words=stop_words)
tfidf_matrix = tf.fit_transform(df_product['processed_description'].values)
tfidf_matrix # la sparse matrix

<2244x9877 sparse matrix of type '<class 'numpy.float64'>'
	with 279250 stored elements in Compressed Sparse Row format>

In [119]:
# cosine_matrix (num_product_description x num_product_description)
# cosine-sim[i, j] = cosin-similarity between product description i and product description j
cosine_sim = linear_kernel(tfidf_matrix.toarray(), tfidf_matrix.toarray()) # tinh toan cosine similarity matrix

In [ ]:
# # Convert the NumPy array to a pandas DataFrame first, then save to CSV
# feature_names = tf.get_feature_names_out()
# pd.DataFrame(feature_names, columns=['feature_tfidf_names']).to_csv('../data/feature_tfidf_names.csv', index=False)

In [120]:
cosine_sim

array([[1.        , 0.85782008, 0.14497161, ..., 0.17827546, 0.16146832,
        0.1760463 ],
       [0.85782008, 1.        , 0.15881265, ..., 0.18823641, 0.18071754,
        0.16985246],
       [0.14497161, 0.15881265, 1.        , ..., 0.1440359 , 0.14132204,
        0.11507099],
       ...,
       [0.17827546, 0.18823641, 0.1440359 , ..., 1.        , 0.214457  ,
        0.22759254],
       [0.16146832, 0.18071754, 0.14132204, ..., 0.214457  , 1.        ,
        0.15593958],
       [0.1760463 , 0.16985246, 0.11507099, ..., 0.22759254, 0.15593958,
        1.        ]])

In [117]:
cosine_sim.shape

(2244, 2244)